# 🗂️ From text to a table
### Extract, clean up and summarise text into fields you can count

**Module 2 · Session 02 · notebook 2 of 3**

Most text in a company never reaches a spreadsheet: contracts, reports, news, emails. An LLM can read each document and
fill in the same fields every time. The moment text becomes a table, you can sort, count, filter and plot it.

This notebook covers three work patterns and one habit:

- **Extract:** text in, a table row out.
- **Clean up:** a messy value in, a standard value out.
- **Summarise to a form:** text in, fixed fields of fixed length out.
- **Check with code:** never trust a field you can verify automatically.

Everything runs on `gemma4:e2b` in your Colab runtime. At the end we send the same work to a
cloud model, so you can compare the two.

**Before you start:** Runtime → Change runtime type → T4 GPU. Then Runtime → Run all.

**Contents**
1. Ask nicely, or enforce a form
2. Extract from a news text
3. Check with code
4. Clean up messy values
5. Summarise to a form
6. A small batch, then a chart
7. ✏️ Design your own schema
8. Same code, cloud model

**Own machine.** This notebook also runs in Jupyter or VS Code on a laptop with Ollama installed: Moodle explains how. Leave `BACKEND` on `ollama`.

**Plan B, if local does not work** (no free GPU, or Ollama fails). Create a free key at [aistudio.google.com](https://aistudio.google.com) and
store it as a Colab secret called `GEMINI_API_KEY` (key icon on the left, then switch on access for this notebook). In the setup cell, set `BACKEND`
to `gemini` and run everything again. A Groq key from [console.groq.com](https://console.groq.com), stored as `GROQ_API_KEY`, works the same way with `groq`.

The code stays the same and your texts go to the provider. Free keys allow few calls a minute and about 500 a day, so everyone needs their own key, and plan B runs on smaller samples and takes longer. Your numbers will differ from the slides, because the cloud model is stronger.

In [ ]:
# @title Setup: install Ollama and download the models (2 to 4 minutes) { display-mode: "form" }
BACKEND = "ollama"  # @param ["ollama", "gemini", "groq"]
%pip install -q -U openai  2> /dev/null
import hashlib, json, os, random, shutil, subprocess, sys, time, warnings
from concurrent.futures import ThreadPoolExecutor
from typing import Literal, Optional
import numpy as np, pandas as pd, matplotlib.pyplot as plt, requests
from openai import OpenAI
from pydantic import BaseModel, Field

warnings.filterwarnings("ignore")

OLLAMA = "http://localhost:11434"
DECIDER = "d3:2b"
DEVICE = "cpu" if sys.platform == "darwin" else None   # on a Mac, Ollama already fills the GPU memory
if BACKEND == "ollama":
    if shutil.which("ollama") is None and "google.colab" in sys.modules:
        subprocess.run("apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)
    if shutil.which("ollama") is None:
        raise RuntimeError("Ollama is not installed. On your own machine, install it as described on Moodle. Otherwise use plan B: see the top of this notebook.")

    # start_new_session keeps the server alive when you stop a cell
    subprocess.Popen(["ollama", "serve"], start_new_session=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(4)
    for model in ["gemma4:e2b"]:
        t0 = time.time()
        done = subprocess.run(["ollama", "pull", model], capture_output=True)
        print(f"{model} ready ({time.time() - t0:.0f} s)" if done.returncode == 0 else f"{model} did not download. Use plan B: see the top of this notebook.")

    if "google.colab" in sys.modules:
        gpu = subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout.strip()
        print(gpu if gpu else "No GPU on this runtime. Each text would take 10 to 20 seconds, which is too slow here. Use Runtime → Change runtime type → T4 GPU. If Colab has none free, use plan B: see the top of this notebook.")
    BASE_URL, API_KEY, MODEL, EXTRA, PAUSE = OLLAMA + "/v1", "ollama", "gemma4:e2b", {"reasoning_effort": "none"}, 0
else:   # plan B: the same calls go to a cloud model. Free keys allow 15 (Gemini) or 30 (Groq) calls a minute
    BASE_URL, secret, MODEL, EXTRA, PAUSE = {
        "gemini": ("https://generativelanguage.googleapis.com/v1beta/openai", "GEMINI_API_KEY", "gemini-flash-lite-latest", {}, 4.2),
        "groq": ("https://api.groq.com/openai/v1", "GROQ_API_KEY", "openai/gpt-oss-20b", {"reasoning_effort": "low"}, 3),
    }[BACKEND]
    try:
        from google.colab import userdata
        API_KEY = userdata.get(secret)
    except ImportError:
        API_KEY = os.environ[secret]

NAVY, LAV, GREY, ORANGE = "#211A52", "#C2C1CC", "#54616E", "#D2542A"
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.edgecolor": GREY, "axes.labelcolor": GREY,
                     "xtick.color": GREY, "ytick.color": GREY, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#E6E6EC", "legend.frameon": False})
pd.set_option("display.max_colwidth", 200)

---
# 📐 1 · Ask nicely, or enforce a form

We start with a real text about venture funding. First we only *ask* for JSON, a format that programs can read.

In [ ]:
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print(BASE_URL, "·", MODEL)

In [ ]:
VC_TEXT = """Fresh PitchBook numbers show how distorted the startup funding landscape has become in 2025. Venture investors have put roughly 192.7 billion US dollars into AI companies this year. Total global venture funding across all sectors is sitting around 366.8 billion dollars. That means more than half of every VC dollar in 2025 is going to AI startups, not fintech, not climate, not biotech, not anything else.

It's even more extreme if you zoom in on the most recent quarter in the United States. In Q3 2025, AI startups captured about 62.7 percent of all U.S. venture money. Globally, the share in that same period is about 53.2 percent. If you pitch yourself as anything other than AI infrastructure, AI agents or AI copilots, good luck closing a round.

Analysts are worried about obvious bubble signals. Seed-stage companies with no revenue are raising at unicorn valuations purely on the promise that they are AI-native. Non-AI founders are rewriting their decks and changing their category labels just to not get ignored.

On the other hand, investors argue this is not a classic dot-com bubble. Fortune 500 buyers are piloting AI and budgeting for 2026, and governments treat AI as critical infrastructure. Still, a single theme absorbing about 200 billion dollars in one year sets off alarms about concentration risk."""

response = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    messages=[{"role": "user", "content": "Return JSON with the keys total_ai_funding and main_risk.\n\n" + VC_TEXT}],
    **EXTRA,
)
answer = response.choices[0].message.content
print(answer)
try:
    print("\nParsed:", json.loads(answer))
except json.JSONDecodeError as e:
    print("\nThis is not valid JSON for a program:", e)

Sometimes that works. Sometimes the model adds a line of chat around the JSON, renames a key or returns a number as
a sentence, and your program stops at row 4,000. Asking is not enough for repeatable work.

With a **schema**, the allowed form is sent along with the request, and the output is built to fit it. We write the schema as a
Pydantic class, as in notebook 1. `Field` adds a description for each value, and `Optional` means the value may be missing.

In [ ]:
class Funding(BaseModel):
    total_ai_funding_usd_billion: float = Field(description="AI venture funding in 2025, in billions of US dollars")
    share_of_all_vc_percent: Optional[float] = Field(description="Share of all venture funding, if stated")
    main_risk: str = Field(description="The main worry, in at most 10 words")

response = client.chat.completions.parse(
    model=MODEL,
    temperature=0,
    messages=[{"role": "system", "content": "Extract the fields from the text. Use only what the text says."},
              {"role": "user", "content": VC_TEXT}],
    response_format=Funding,
    **EXTRA,
)

funding = response.choices[0].message.parsed
print(funding)
print(funding.total_ai_funding_usd_billion, "is now a number you can calculate with")

We will make many calls like this, so we take the `ask()` function you built in notebook 1: the same call, with a retry.

In [ ]:
def ask(system, user, schema):
    for temperature in (0, 0.2, 0.4):   # the same temperature would repeat the same mistake
        time.sleep(PAUSE)
        try:
            response = client.chat.completions.parse(
                model=MODEL,
                temperature=temperature,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
                response_format=schema,
                **EXTRA,
            )
            return response.choices[0].message.parsed
        except Exception as e:
            error = e
            if "429" in str(e):
                time.sleep(30)   # over the free limit: wait for the next minute
    raise error

---
# 📰 2 · Extract from a news text

A longer form on a second text. Remember notebook 1: on Ollama the schema limits what the model may answer, but the model does not
read it. So the `Field` descriptions have to go into the instruction too. We add them with a plain loop. Good descriptions read like clear
instructions to a new colleague.

In [ ]:
INFRA_TEXT = """Researchers at MIT are warning that the current strategy of simply making AI models bigger and throwing more compute at them may already be running into diminishing returns. They studied how recent frontier models scale versus how their efficiency improves, and they argue that in many areas, especially reasoning and language tasks, the performance curve is flattening out for giant models.

The paper points to the fact that relatively lean models now deliver output that used to require huge GPU farms. An example is the wave of cheap, efficient models, such as DeepSeek's low-cost model from January 2025, which shocked a lot of people in Silicon Valley.

That creates a tension with what the biggest US players are doing. OpenAI and other hyperscalers are signing long-horizon data center and energy deals worth hundreds of billions of dollars. Economists quoted in the reporting say this looks like a speculative bubble: the capital intensity is enormous, the timelines are long, and it is not obvious that enterprises will pay enough for AI assistants and agents to justify the spend.

If the huge-bet strategy is wrong, taxpayers, regulators and energy grids are still on the hook, because these data centers tie up land, electricity and water at national scale."""

class Article(BaseModel):
    summary: str = Field(description="Two neutral sentences on the main point")
    institutions: list[str] = Field(description="Organisations and companies named in the text")
    economic_scale: Optional[str] = Field(description="The largest money amount, quoted from the text")
    risks: list[str] = Field(description="Warnings in the text, at most 3, each at most 10 words")
    evidence: str = Field(description="One sentence copied word for word from the text that supports the summary")

instruction = "Extract the fields. Use only what the text says.\n"
for name, field in Article.model_fields.items():
    instruction += f"{name}: {field.description}\n"
print(instruction)

art = ask(instruction, INFRA_TEXT, Article)
print(art.model_dump_json(indent=2))

---
# ✅ 3 · Check with code

The schema guarantees the **form**. It does not guarantee the **content**: the fields can still be wrong. Some content
you can check without reading anything. We asked for an exact quote, so code can test whether the quote really is in the
text. The same goes for numbers and names.

In [ ]:
print(art.evidence in INFRA_TEXT)

That test is strict: one capital letter or line break, and an exact quote fails. So we compare lower-case text with single spaces.

In [ ]:
def normal(text):
    return " ".join(text.lower().split())

print("quote is in the text:", normal(art.evidence) in normal(INFRA_TEXT))
for name in art.institutions:
    print(f"{name:<20} in the text: {normal(name) in normal(INFRA_TEXT)}")

---
# 🧹 4 · Clean up messy values

Real data has the same thing written many ways. This job is dull, there is a lot of it, and it is a good fit for a small
model: short input, short output, and an answer you can check. `Literal` restricts the answer to our list.

In [ ]:
MESSY = ["Novo Nordisk A/S", "NOVO", "novo nordisk", "Novo-Nordisk Pharma", "Maersk", "A.P. Moller - Maersk", "A P Møller",
         "Mærsk Line", "Lego", "THE LEGO GROUP", "LEGO System A/S", "Carlsberg Breweries", "carlsberg", "Tuborg (Carlsberg)",
         "Danske Bank A/S", "DANSKE", "Vestas Wind Systems", "VESTAS", "IKEA Denmark"]
COMPANIES = ["Novo Nordisk", "Maersk", "Lego", "Carlsberg", "Danske Bank", "Vestas", "other"]

class Clean(BaseModel):
    company: Literal[tuple(COMPANIES)]

rows = []
for name in MESSY:
    answer = ask("Match this name to one company on this list: " + ", ".join(COMPANIES), name, Clean)
    rows.append({"raw": name, "company": answer.company})

pd.DataFrame(rows)

🤔 *`IKEA Denmark` is not on the list. Did the model answer "other"? Forcing a choice from a list always gives an answer,
so give the model a way out.*

---
# 📄 5 · Summarise to a form

A free summary is hard to compare across documents. A form is easy. Think of a hundred reports, each in the same three
lines.

We need the loop over the fields again, so it becomes a short function.

In [ ]:
def with_fields(instruction, schema):
    for name, field in schema.model_fields.items():
        if field.description:
            instruction += f"\n{name}: {field.description}"
    return instruction

class Brief(BaseModel):
    topic: Literal["funding", "infrastructure", "regulation", "products", "other"]
    claim: str = Field(description="The main claim in at most 15 words")
    who_should_care: str = Field(description="Which kind of reader should care, in at most 6 words")

rows = []
for text in [VC_TEXT, INFRA_TEXT]:
    answer = ask(with_fields("Summarise the text into the form.", Brief), text, Brief)
    rows.append(answer.model_dump())

pd.DataFrame(rows)

---
# 📊 6 · A small batch, then a chart

From one document to many. We use 60 passages from earnings calls, the quarterly calls where company managers talk to
investors, each one mentioning AI. Is the passage a result that already happened, a plan, or just enthusiasm?

Three habits for batches:
- Start small, and look at the first results before running the rest.
- Save as you go, so a crash costs you nothing.
- Measure the time per row, and multiply before you commit to the full run.

In [ ]:
passages = pd.read_parquet("https://raw.githubusercontent.com/aaubs/ds-master/main/data/earnings_ai/passages.parquet")
batch = passages.sample(60 if BACKEND == "ollama" else 20, random_state=11).reset_index(drop=True)

class Claim(BaseModel):
    use_case: str = Field(description="What the AI is used for, at most 6 plain words, or 'not stated'")
    claim_type: Literal["concrete_result", "plan", "vague_hype"]
    evidence: str = Field(description="The shortest exact quote from the passage that justifies claim_type")

SYSTEM = '''You read passages from company earnings calls in which someone mentions AI.
claim_type is one of:
concrete_result: something that already happened, with a specific outcome such as revenue, customers, or time saved.
plan: a stated intention, investment or launch that has not produced a result yet.
vague_hype: enthusiasm or positioning with no specific outcome or plan.
Excitement is not a result. Forecasts are plan or vague_hype.'''
SYSTEM = with_fields(SYSTEM, Claim)
print(SYSTEM)

answer = ask(SYSTEM, batch["text"].iloc[0], Claim)
print(batch["text"].iloc[0][:300])
print(answer)

That looks right, so we run the batch.

In [ ]:
rows = []
t0 = time.time()
for i, p in enumerate(batch.itertuples()):
    answer = ask(SYSTEM, p.text, Claim)
    rows.append({"company": p.company, "year": p.year, "text": p.text, **answer.model_dump()})
    if i % 20 == 19:
        pd.DataFrame(rows).to_csv("claims.csv", index=False)   # save as you go
        print(i + 1, "done,", round((time.time() - t0) / (i + 1), 2), "s per passage")

claims = pd.DataFrame(rows)
per_row = (time.time() - t0) / len(claims)
print(f"All {len(passages):,} passages would take {per_row * len(passages) / 60:.0f} minutes at this speed.")
claims[["company", "use_case", "claim_type", "evidence"]].head(8)

In [ ]:
# @title Chart: what kind of claims? { display-mode: "form" }
claims["quote_ok"] = [normal(e) in normal(t) for e, t in zip(claims["evidence"], claims["text"])]
print(f"Quotes found in the passage: {claims['quote_ok'].mean():.0%}")
order = ["concrete_result", "plan", "vague_hype"]
shares = claims["claim_type"].value_counts(normalize=True).reindex(order).fillna(0)
fig, ax = plt.subplots(figsize=(6, 2.2))
ax.barh(order[::-1], shares[order[::-1]] * 100, color=[LAV, GREY, NAVY])
ax.set_xlabel(f"% of {len(claims)} passages"); ax.grid(axis="y", visible=False)
plt.show()

✏️ *Open `claims` and read ten rows against the text, as you did in notebook 1. Where is the model too generous? Would you
trust this table for a report to a client?*

The extension notebook **Who is bluffing about AI?** on Moodle takes this same idea to all passages, by teaching a cheap
classifier from a few hundred LLM labels. Notebook 3 shows why that is the better way to scale.

---
# ✏️ 7 · Design your own schema

Pick one of these and write the class, then run it on a text you paste in:
- **Job ad:** company, role, must-have skills, salary if stated, remote or not.
- **Product review:** product, rating out of 5, pros, cons, would they recommend it.
- **News item:** who, did what, when, how much.

Field descriptions matter more than field names. Allow `Optional` when a value may be missing, or the model will make one up.

In [ ]:
MY_TEXT = """Paste a text here."""

class MySchema(BaseModel):
    # ✏️ your fields
    summary: str = Field(description="One sentence")

ask(with_fields("Extract the fields. Use only what the text says.", MySchema), MY_TEXT, MySchema)

---
# ☁️ 8 · Same code, cloud model

Everything so far ran on your own runtime: no key, no cost per call, and the text stays with you. A cloud model is usually
stronger and needs no GPU, but it has limits per day, and your text goes to another company.

Because Ollama and the big providers share one format, moving takes two changes: the `client` and the model name. To try it, create a free
key at [aistudio.google.com](https://aistudio.google.com) and store it as a Colab secret called `GEMINI_API_KEY` (key icon on
the left). This part is optional: without a key the cell just tells you so.

In [ ]:
try:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
except Exception:
    key = os.environ.get("GEMINI_API_KEY")

if BACKEND != "ollama":
    print("You are already on the cloud route, so there is nothing to compare here.")
elif not key:
    print("No key found: skipping the cloud comparison.")
else:
    cloud = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai", api_key=key)   # another address, your own key

    t0 = time.time()
    for p in batch.head(5).itertuples():
        response = cloud.chat.completions.parse(
            model="gemini-flash-lite-latest",
            temperature=0,
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user", "content": p.text}],
            response_format=Claim,
        )
        print(response.choices[0].message.parsed.claim_type)

    print(f"cloud: {(time.time() - t0) / 5:.2f} s per passage · your runtime: {per_row:.2f} s per passage")

| | Your own runtime | Cloud API |
|---|---|---|
| Account and key | none | needed |
| Cost per call | none | small, with a free tier |
| Limits | your GPU | requests per minute and per day |
| Where the text goes | stays with you | to the provider |
| Model strength | small models | strong models |

🤔 *You have 5,000 customer emails with personal details. Which column do you pick, and what changes if you have 50 long
contracts and need the best quality?*

**Next:** notebook 3. An LLM writes every answer word by word, and that is slow when you only need a decision.